# QQQ TEMA Strategy with Boruta Feature Selection
## Find the Best Indicator to Ensemble with Triple EMA

This notebook:
1. Tests your base TEMA (Triple EMA) strategy on QQQ
2. Computes signals for multiple auxiliary indicators
3. Uses **Boruta-style validation** (Sharpe-optimized) to identify statistically significant indicators
4. Tests ensemble combinations to find the best pairing with TEMA

**Boruta Adaptation for Quant:**
Instead of shuffling features, we shuffle returns and compare the real strategy Sharpe against shuffled Sharpes.
If the real Sharpe beats the shuffled versions consistently, the indicator has predictive value.

In [ ]:
# ==================== CELL 1: ENVIRONMENT SETUP ====================
!pip install yfinance TA-Lib numpy pandas vectorbt scikit-learn -q

import numpy as np
import pandas as pd
import yfinance as yf
import talib
import vectorbt as vbt
from datetime import datetime, timedelta
import warnings
import itertools
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Environment Setup Complete")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")

In [ ]:
# ==================== CELL 2: DOWNLOAD QQQ DATA ====================

TICKER = 'QQQ'
START_DATE = '2018-01-01'
END_DATE = datetime.now().strftime('%Y-%m-%d')

# Download data
print(f"📥 Downloading {TICKER} data from {START_DATE} to {END_DATE}...")
data = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)

# Flatten MultiIndex columns if present
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Ensure we have the required columns
data = data[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

print(f"\n✅ Data downloaded successfully!")
print(f"   Date Range: {data.index[0].strftime('%Y-%m-%d')} to {data.index[-1].strftime('%Y-%m-%d')}")
print(f"   Total Bars: {len(data)}")
print(f"\n📊 Sample Data:")
data.tail()

In [ ]:
# ==================== CELL 3: DEFINE IN-SAMPLE / OUT-OF-SAMPLE SPLIT ====================

# 60% In-Sample, 40% Out-of-Sample (standard for walk-forward)
IS_RATIO = 0.60

split_idx = int(len(data) * IS_RATIO)
is_data = data.iloc[:split_idx].copy()
oos_data = data.iloc[split_idx:].copy()

print("="*70)
print("📊 DATA SPLIT CONFIGURATION")
print("="*70)
print(f"\n🔵 IN-SAMPLE (Training):")
print(f"   Period: {is_data.index[0].strftime('%Y-%m-%d')} to {is_data.index[-1].strftime('%Y-%m-%d')}")
print(f"   Bars: {len(is_data)} ({IS_RATIO*100:.0f}%)")
print(f"\n🟢 OUT-OF-SAMPLE (Validation):")
print(f"   Period: {oos_data.index[0].strftime('%Y-%m-%d')} to {oos_data.index[-1].strftime('%Y-%m-%d')}")
print(f"   Bars: {len(oos_data)} ({(1-IS_RATIO)*100:.0f}%)")

In [ ]:
# ==================== CELL 4: INDICATOR CALCULATION FUNCTIONS ====================

def calc_tema_signals(close, ema1_period, ema2_period, ema3_period):
    """
    Triple EMA Crossover Strategy
    LONG when: EMA1 > EMA2 AND EMA1 > EMA3
    EXIT when: EMA1 < EMA2 OR EMA1 < EMA3
    """
    ema1 = talib.EMA(close, timeperiod=ema1_period)
    ema2 = talib.EMA(close, timeperiod=ema2_period)
    ema3 = talib.EMA(close, timeperiod=ema3_period)
    
    bullish = (ema1 > ema2) & (ema1 > ema3)
    bullish_prev = pd.Series(bullish).shift(1).fillna(False).values
    
    entries = (~bullish_prev) & bullish
    exits = bullish_prev & (~bullish)
    
    return pd.Series(entries, index=close.index), pd.Series(exits, index=close.index)


def calc_macd_signals(close, fast=12, slow=26, signal=9):
    """
    MACD Crossover Strategy
    LONG when: MACD crosses above Signal line
    EXIT when: MACD crosses below Signal line
    """
    macd, macd_signal, _ = talib.MACD(close, fastperiod=fast, slowperiod=slow, signalperiod=signal)
    
    bullish = macd > macd_signal
    bullish_prev = pd.Series(bullish).shift(1).fillna(False).values
    
    entries = (~bullish_prev) & bullish
    exits = bullish_prev & (~bullish)
    
    return pd.Series(entries, index=close.index), pd.Series(exits, index=close.index)


def calc_rsi_signals(close, period=14, oversold=30, overbought=70):
    """
    RSI Mean Reversion Strategy
    LONG when: RSI crosses above oversold level
    EXIT when: RSI crosses above overbought level
    """
    rsi = talib.RSI(close, timeperiod=period)
    rsi_prev = pd.Series(rsi).shift(1).fillna(50).values
    
    entries = (rsi_prev <= oversold) & (rsi > oversold)
    exits = (rsi_prev < overbought) & (rsi >= overbought)
    
    return pd.Series(entries, index=close.index), pd.Series(exits, index=close.index)


def calc_supertrend_signals(high, low, close, period=10, multiplier=3.0):
    """
    Supertrend Indicator
    Uses ATR-based trailing stop
    """
    atr = talib.ATR(high, low, close, timeperiod=period)
    hl2 = (high + low) / 2
    
    upper_band = hl2 + multiplier * atr
    lower_band = hl2 - multiplier * atr
    
    supertrend = np.zeros(len(close))
    direction = np.ones(len(close))  # 1 = uptrend, -1 = downtrend
    
    for i in range(1, len(close)):
        if close.iloc[i] > upper_band.iloc[i-1]:
            direction[i] = 1
        elif close.iloc[i] < lower_band.iloc[i-1]:
            direction[i] = -1
        else:
            direction[i] = direction[i-1]
            
        if direction[i] == 1:
            supertrend[i] = max(lower_band.iloc[i], supertrend[i-1]) if direction[i-1] == 1 else lower_band.iloc[i]
        else:
            supertrend[i] = min(upper_band.iloc[i], supertrend[i-1]) if direction[i-1] == -1 else upper_band.iloc[i]
    
    direction_series = pd.Series(direction, index=close.index)
    direction_prev = direction_series.shift(1).fillna(1)
    
    entries = (direction_prev == -1) & (direction_series == 1)
    exits = (direction_prev == 1) & (direction_series == -1)
    
    return entries, exits


def calc_aroon_signals(high, low, period=25):
    """
    Aroon Indicator
    LONG when: Aroon Up crosses above Aroon Down
    EXIT when: Aroon Down crosses above Aroon Up
    """
    aroon_down, aroon_up = talib.AROON(high, low, timeperiod=period)
    
    bullish = aroon_up > aroon_down
    bullish_prev = pd.Series(bullish).shift(1).fillna(False).values
    
    entries = (~bullish_prev) & bullish
    exits = bullish_prev & (~bullish)
    
    return pd.Series(entries, index=high.index), pd.Series(exits, index=high.index)


def calc_adx_signals(high, low, close, period=14, threshold=25):
    """
    ADX Trend Strength with DI Crossover
    LONG when: +DI > -DI AND ADX > threshold
    EXIT when: -DI > +DI
    """
    adx = talib.ADX(high, low, close, timeperiod=period)
    plus_di = talib.PLUS_DI(high, low, close, timeperiod=period)
    minus_di = talib.MINUS_DI(high, low, close, timeperiod=period)
    
    bullish = (plus_di > minus_di) & (adx > threshold)
    bullish_prev = pd.Series(bullish).shift(1).fillna(False).values
    
    entries = (~bullish_prev) & bullish
    exits = bullish_prev & (~bullish)
    
    return pd.Series(entries, index=close.index), pd.Series(exits, index=close.index)


def calc_bbands_signals(close, period=20, std_dev=2.0):
    """
    Bollinger Bands Mean Reversion
    LONG when: Price touches lower band
    EXIT when: Price touches upper band
    """
    upper, middle, lower = talib.BBANDS(close, timeperiod=period, nbdevup=std_dev, nbdevdn=std_dev)
    
    close_prev = pd.Series(close).shift(1).fillna(close.iloc[0]).values
    
    entries = (close_prev > lower) & (close <= lower)
    exits = (close_prev < upper) & (close >= upper)
    
    return pd.Series(entries, index=close.index), pd.Series(exits, index=close.index)


def calc_stoch_signals(high, low, close, fastk=14, slowk=3, slowd=3, oversold=20, overbought=80):
    """
    Stochastic Oscillator
    LONG when: %K crosses above oversold from below
    EXIT when: %K crosses above overbought
    """
    slowk, slowd = talib.STOCH(high, low, close, fastk_period=fastk, slowk_period=slowk, slowd_period=slowd)
    
    slowk_prev = pd.Series(slowk).shift(1).fillna(50).values
    
    entries = (slowk_prev <= oversold) & (slowk > oversold)
    exits = (slowk_prev < overbought) & (slowk >= overbought)
    
    return pd.Series(entries, index=close.index), pd.Series(exits, index=close.index)


def calc_cci_signals(high, low, close, period=20, oversold=-100, overbought=100):
    """
    Commodity Channel Index
    LONG when: CCI crosses above oversold
    EXIT when: CCI crosses above overbought
    """
    cci = talib.CCI(high, low, close, timeperiod=period)
    cci_prev = pd.Series(cci).shift(1).fillna(0).values
    
    entries = (cci_prev <= oversold) & (cci > oversold)
    exits = (cci_prev < overbought) & (cci >= overbought)
    
    return pd.Series(entries, index=close.index), pd.Series(exits, index=close.index)


print("✅ Indicator Functions Defined:")
print("   1. TEMA (Triple EMA Crossover)")
print("   2. MACD (Crossover)")
print("   3. RSI (Oversold/Overbought)")
print("   4. Supertrend (ATR-based)")
print("   5. Aroon (Up/Down Crossover)")
print("   6. ADX (Trend Strength + DI)")
print("   7. Bollinger Bands (Mean Reversion)")
print("   8. Stochastic (Oversold/Overbought)")
print("   9. CCI (Commodity Channel Index)")

In [ ]:
# ==================== CELL 5: GENERATE ALL INDICATOR SIGNALS ====================

# Base TEMA parameters (from your grid search - best performers)
TEMA_PARAMS = {'ema1': 4, 'ema2': 149, 'ema3': 106}

# Calculate all indicator signals on FULL data
close = data['Close'].values
high = data['High'].values  
low = data['Low'].values
idx = data.index

print("="*70)
print("📊 GENERATING INDICATOR SIGNALS")
print("="*70)

# Store all indicator signals
indicator_signals = {}

# 1. TEMA (Base Strategy)
ent, exi = calc_tema_signals(pd.Series(close), TEMA_PARAMS['ema1'], TEMA_PARAMS['ema2'], TEMA_PARAMS['ema3'])
indicator_signals['TEMA'] = {'entries': ent.values, 'exits': exi.values, 'params': TEMA_PARAMS}
print(f"✓ TEMA({TEMA_PARAMS['ema1']},{TEMA_PARAMS['ema2']},{TEMA_PARAMS['ema3']}): {ent.sum()} entry signals")

# 2. MACD
ent, exi = calc_macd_signals(pd.Series(close), fast=12, slow=26, signal=9)
indicator_signals['MACD'] = {'entries': ent.values, 'exits': exi.values, 'params': {'fast':12, 'slow':26, 'signal':9}}
print(f"✓ MACD(12,26,9): {ent.sum()} entry signals")

# 3. RSI
ent, exi = calc_rsi_signals(pd.Series(close), period=14, oversold=30, overbought=70)
indicator_signals['RSI'] = {'entries': ent.values, 'exits': exi.values, 'params': {'period':14, 'oversold':30, 'overbought':70}}
print(f"✓ RSI(14,30,70): {ent.sum()} entry signals")

# 4. Supertrend
ent, exi = calc_supertrend_signals(pd.Series(high), pd.Series(low), pd.Series(close), period=10, multiplier=3.0)
indicator_signals['SUPERTREND'] = {'entries': ent.values, 'exits': exi.values, 'params': {'period':10, 'mult':3.0}}
print(f"✓ SUPERTREND(10,3.0): {ent.sum()} entry signals")

# 5. Aroon
ent, exi = calc_aroon_signals(pd.Series(high), pd.Series(low), period=25)
indicator_signals['AROON'] = {'entries': ent.values, 'exits': exi.values, 'params': {'period':25}}
print(f"✓ AROON(25): {ent.sum()} entry signals")

# 6. ADX
ent, exi = calc_adx_signals(pd.Series(high), pd.Series(low), pd.Series(close), period=14, threshold=25)
indicator_signals['ADX'] = {'entries': ent.values, 'exits': exi.values, 'params': {'period':14, 'threshold':25}}
print(f"✓ ADX(14,25): {ent.sum()} entry signals")

# 7. Bollinger Bands
ent, exi = calc_bbands_signals(pd.Series(close), period=20, std_dev=2.0)
indicator_signals['BBANDS'] = {'entries': ent.values, 'exits': exi.values, 'params': {'period':20, 'std_dev':2.0}}
print(f"✓ BBANDS(20,2.0): {ent.sum()} entry signals")

# 8. Stochastic
ent, exi = calc_stoch_signals(pd.Series(high), pd.Series(low), pd.Series(close))
indicator_signals['STOCH'] = {'entries': ent.values, 'exits': exi.values, 'params': {'fastk':14, 'slowk':3, 'slowd':3}}
print(f"✓ STOCH(14,3,3): {ent.sum()} entry signals")

# 9. CCI
ent, exi = calc_cci_signals(pd.Series(high), pd.Series(low), pd.Series(close), period=20)
indicator_signals['CCI'] = {'entries': ent.values, 'exits': exi.values, 'params': {'period':20}}
print(f"✓ CCI(20): {ent.sum()} entry signals")

print(f"\n✅ Generated signals for {len(indicator_signals)} indicators")

In [ ]:
# ==================== CELL 6: BORUTA VALIDATION FUNCTIONS ====================

def calculate_sharpe(returns, periods_per_year=252):
    """
    Calculate annualized Sharpe ratio
    For equities like QQQ, use 252 trading days
    """
    if len(returns) == 0 or returns.std() == 0:
        return -999
    return (returns.mean() / returns.std()) * np.sqrt(periods_per_year)


def run_backtest(close_prices, entries, exits, init_cash=10000, fees=0.001):
    """
    Run vectorbt backtest and return portfolio returns
    """
    try:
        pf = vbt.Portfolio.from_signals(
            close_prices,
            entries=entries,
            exits=exits,
            init_cash=init_cash,
            fees=fees,
            freq='D'
        )
        return pf.returns(), pf
    except:
        return pd.Series([0.0]), None


def boruta_validate_sharpe(oos_returns, n_iterations=100, confidence_threshold=0.95):
    """
    Boruta-style validation for Sharpe ratio
    
    Compares real OOS Sharpe against shuffled returns.
    If real Sharpe beats shuffled versions >= confidence_threshold% of the time,
    the indicator is considered statistically significant.
    
    Returns:
        boruta_score: Percentage of times real Sharpe beat shuffled (0-100)
        is_confirmed: True if boruta_score >= confidence_threshold*100
        real_sharpe: The actual OOS Sharpe ratio
    """
    oos_clean = oos_returns.dropna()
    if len(oos_clean) < 20:  # Need minimum observations
        return 0, False, -999
    
    real_sharpe = calculate_sharpe(oos_clean)
    
    shadow_wins = 0
    for _ in range(n_iterations):
        # Shuffle returns (destroy temporal structure)
        shuffled = np.random.permutation(oos_clean.values)
        shuffled_sharpe = calculate_sharpe(pd.Series(shuffled))
        
        if real_sharpe > shuffled_sharpe:
            shadow_wins += 1
    
    boruta_score = (shadow_wins / n_iterations) * 100
    is_confirmed = boruta_score >= (confidence_threshold * 100)
    
    return boruta_score, is_confirmed, real_sharpe


print("✅ Boruta Validation Functions Defined")
print("   - calculate_sharpe(): Annualized Sharpe (252 days for equities)")
print("   - run_backtest(): VectorBT portfolio simulation")
print("   - boruta_validate_sharpe(): Statistical significance test")

In [ ]:
# ==================== CELL 7: INDIVIDUAL INDICATOR BORUTA TEST ====================

print("="*100)
print("🔬 BORUTA FEATURE SELECTION: INDIVIDUAL INDICATOR ANALYSIS")
print("="*100)
print(f"\nTesting each indicator independently with Boruta validation (100 shuffle iterations)...\n")

individual_results = []

for name, signals in tqdm(indicator_signals.items(), desc="Testing Indicators"):
    entries = pd.Series(signals['entries'], index=data.index)
    exits = pd.Series(signals['exits'], index=data.index)
    
    # Apply 1-bar execution delay (realistic execution)
    entries_delayed = entries.shift(1).fillna(False).astype(bool)
    exits_delayed = exits.shift(1).fillna(False).astype(bool)
    
    # Full period backtest
    returns, pf = run_backtest(data['Close'], entries_delayed, exits_delayed)
    
    if pf is None:
        continue
    
    # Split returns for IS/OOS
    is_returns = returns.iloc[:split_idx]
    oos_returns = returns.iloc[split_idx:]
    
    # Calculate metrics
    is_sharpe = calculate_sharpe(is_returns)
    oos_sharpe = calculate_sharpe(oos_returns)
    
    # Boruta validation on OOS returns
    boruta_score, is_confirmed, _ = boruta_validate_sharpe(oos_returns, n_iterations=100)
    
    # Portfolio stats
    total_return = pf.total_return()
    max_dd = pf.max_drawdown()
    num_trades = pf.trades.count()
    win_rate = pf.trades.win_rate() if num_trades > 0 else 0
    
    individual_results.append({
        'Indicator': name,
        'IS_Sharpe': is_sharpe,
        'OOS_Sharpe': oos_sharpe,
        'Sharpe_Decay': ((oos_sharpe - is_sharpe) / abs(is_sharpe) * 100) if is_sharpe != 0 else 0,
        'Boruta_Score': boruta_score,
        'Boruta_Confirmed': '✅ YES' if is_confirmed else '❌ NO',
        'Total_Return': total_return,
        'Max_DD': max_dd,
        'Num_Trades': num_trades,
        'Win_Rate': win_rate
    })

# Create DataFrame and sort by Boruta Score
individual_df = pd.DataFrame(individual_results)
individual_df = individual_df.sort_values('Boruta_Score', ascending=False).reset_index(drop=True)

# Format for display
display_df = individual_df.copy()
display_df['IS_Sharpe'] = display_df['IS_Sharpe'].map('{:.3f}'.format)
display_df['OOS_Sharpe'] = display_df['OOS_Sharpe'].map('{:.3f}'.format)
display_df['Sharpe_Decay'] = display_df['Sharpe_Decay'].map('{:.1f}%'.format)
display_df['Boruta_Score'] = display_df['Boruta_Score'].map('{:.1f}%'.format)
display_df['Total_Return'] = display_df['Total_Return'].map('{:.2%}'.format)
display_df['Max_DD'] = display_df['Max_DD'].map('{:.2%}'.format)
display_df['Win_Rate'] = display_df['Win_Rate'].map('{:.1%}'.format)

print("\n" + "="*100)
print("📊 INDIVIDUAL INDICATOR RESULTS (Ranked by Boruta Score)")
print("="*100)
print(display_df.to_string(index=False))

# Identify confirmed indicators
confirmed = individual_df[individual_df['Boruta_Confirmed'] == '✅ YES']['Indicator'].tolist()
print(f"\n✅ BORUTA CONFIRMED INDICATORS ({len(confirmed)}): {', '.join(confirmed) if confirmed else 'None'}")
print("   (These indicators have statistically significant predictive power)")

In [ ]:
# ==================== CELL 8: ENSEMBLE COMBINATIONS WITH TEMA ====================

print("="*100)
print("🔗 ENSEMBLE ANALYSIS: FINDING BEST INDICATOR TO PAIR WITH TEMA")
print("="*100)

# Get TEMA signals as base
tema_entries = pd.Series(indicator_signals['TEMA']['entries'], index=data.index).shift(1).fillna(False).astype(bool)
tema_exits = pd.Series(indicator_signals['TEMA']['exits'], index=data.index).shift(1).fillna(False).astype(bool)

# Test TEMA alone first
tema_returns, tema_pf = run_backtest(data['Close'], tema_entries, tema_exits)
tema_is_returns = tema_returns.iloc[:split_idx]
tema_oos_returns = tema_returns.iloc[split_idx:]
tema_is_sharpe = calculate_sharpe(tema_is_returns)
tema_oos_sharpe = calculate_sharpe(tema_oos_returns)
tema_boruta, tema_confirmed, _ = boruta_validate_sharpe(tema_oos_returns)

print(f"\n📍 BASE STRATEGY: TEMA({TEMA_PARAMS['ema1']},{TEMA_PARAMS['ema2']},{TEMA_PARAMS['ema3']})")
print(f"   IS Sharpe: {tema_is_sharpe:.3f}")
print(f"   OOS Sharpe: {tema_oos_sharpe:.3f}")
print(f"   Boruta Score: {tema_boruta:.1f}% {'✅' if tema_confirmed else '❌'}")

# Other indicators to ensemble with TEMA
other_indicators = [k for k in indicator_signals.keys() if k != 'TEMA']

ensemble_results = []

print(f"\n🔬 Testing TEMA + [X] Ensembles (OR Logic)...\n")

for ind_name in tqdm(other_indicators, desc="Testing Ensembles"):
    ind_entries = pd.Series(indicator_signals[ind_name]['entries'], index=data.index).shift(1).fillna(False).astype(bool)
    ind_exits = pd.Series(indicator_signals[ind_name]['exits'], index=data.index).shift(1).fillna(False).astype(bool)
    
    # OR Logic: Entry if EITHER indicator fires
    ensemble_entries = tema_entries | ind_entries
    ensemble_exits = tema_exits | ind_exits
    
    # Run backtest
    returns, pf = run_backtest(data['Close'], ensemble_entries, ensemble_exits)
    
    if pf is None:
        continue
    
    is_returns = returns.iloc[:split_idx]
    oos_returns = returns.iloc[split_idx:]
    
    is_sharpe = calculate_sharpe(is_returns)
    oos_sharpe = calculate_sharpe(oos_returns)
    boruta_score, is_confirmed, _ = boruta_validate_sharpe(oos_returns)
    
    # Calculate improvement over TEMA alone
    sharpe_improvement = ((oos_sharpe - tema_oos_sharpe) / abs(tema_oos_sharpe) * 100) if tema_oos_sharpe != 0 else 0
    
    ensemble_results.append({
        'Ensemble': f'TEMA OR {ind_name}',
        'IS_Sharpe': is_sharpe,
        'OOS_Sharpe': oos_sharpe,
        'vs_TEMA_Alone': sharpe_improvement,
        'Boruta_Score': boruta_score,
        'Boruta_Confirmed': '✅ YES' if is_confirmed else '❌ NO',
        'Total_Return': pf.total_return(),
        'Max_DD': pf.max_drawdown(),
        'Num_Trades': pf.trades.count(),
        'Win_Rate': pf.trades.win_rate()
    })

# Sort by OOS Sharpe (primary) and Boruta Score (secondary)
ensemble_df = pd.DataFrame(ensemble_results)
ensemble_df = ensemble_df.sort_values(['OOS_Sharpe', 'Boruta_Score'], ascending=[False, False]).reset_index(drop=True)

# Format for display
display_ens = ensemble_df.copy()
display_ens['IS_Sharpe'] = display_ens['IS_Sharpe'].map('{:.3f}'.format)
display_ens['OOS_Sharpe'] = display_ens['OOS_Sharpe'].map('{:.3f}'.format)
display_ens['vs_TEMA_Alone'] = display_ens['vs_TEMA_Alone'].map('{:+.1f}%'.format)
display_ens['Boruta_Score'] = display_ens['Boruta_Score'].map('{:.1f}%'.format)
display_ens['Total_Return'] = display_ens['Total_Return'].map('{:.2%}'.format)
display_ens['Max_DD'] = display_ens['Max_DD'].map('{:.2%}'.format)
display_ens['Win_Rate'] = display_ens['Win_Rate'].map('{:.1%}'.format)

print("\n" + "="*100)
print("📊 TEMA + [X] ENSEMBLE RESULTS (Ranked by OOS Sharpe)")
print("="*100)
print(display_ens.to_string(index=False))

In [ ]:
# ==================== CELL 9: MULTI-INDICATOR ENSEMBLE COMBINATIONS ====================

print("="*100)
print("🔗 ADVANCED: MULTI-INDICATOR ENSEMBLE COMBINATIONS")
print("="*100)
print("Testing combinations of 2-3 indicators paired with TEMA...\n")

# Use only Boruta-confirmed or top performers for efficiency
# Select top 5 by Boruta score (excluding TEMA)
top_indicators = individual_df[individual_df['Indicator'] != 'TEMA'].nlargest(5, 'Boruta_Score')['Indicator'].tolist()
print(f"🎯 Top indicators for ensemble testing: {', '.join(top_indicators)}\n")

multi_ensemble_results = []

# Test all 2-indicator combinations with TEMA
for r in range(1, min(4, len(top_indicators) + 1)):  # 1, 2, or 3 additional indicators
    for combo in itertools.combinations(top_indicators, r):
        # Start with TEMA
        ens_entries = tema_entries.copy()
        ens_exits = tema_exits.copy()
        
        # OR in each additional indicator
        for ind_name in combo:
            ind_entries = pd.Series(indicator_signals[ind_name]['entries'], index=data.index).shift(1).fillna(False).astype(bool)
            ind_exits = pd.Series(indicator_signals[ind_name]['exits'], index=data.index).shift(1).fillna(False).astype(bool)
            ens_entries = ens_entries | ind_entries
            ens_exits = ens_exits | ind_exits
        
        # Run backtest
        returns, pf = run_backtest(data['Close'], ens_entries, ens_exits)
        
        if pf is None:
            continue
        
        is_returns = returns.iloc[:split_idx]
        oos_returns = returns.iloc[split_idx:]
        
        is_sharpe = calculate_sharpe(is_returns)
        oos_sharpe = calculate_sharpe(oos_returns)
        boruta_score, is_confirmed, _ = boruta_validate_sharpe(oos_returns)
        
        sharpe_improvement = ((oos_sharpe - tema_oos_sharpe) / abs(tema_oos_sharpe) * 100) if tema_oos_sharpe != 0 else 0
        
        multi_ensemble_results.append({
            'Ensemble': 'TEMA OR ' + ' OR '.join(combo),
            'Num_Indicators': len(combo) + 1,
            'IS_Sharpe': is_sharpe,
            'OOS_Sharpe': oos_sharpe,
            'vs_TEMA_Alone': sharpe_improvement,
            'Boruta_Score': boruta_score,
            'Boruta_Confirmed': '✅' if is_confirmed else '❌',
            'Total_Return': pf.total_return(),
            'Max_DD': pf.max_drawdown(),
            'Num_Trades': pf.trades.count(),
            'Win_Rate': pf.trades.win_rate()
        })

# Sort by OOS Sharpe
multi_df = pd.DataFrame(multi_ensemble_results)
multi_df = multi_df.sort_values('OOS_Sharpe', ascending=False).reset_index(drop=True)

# Display top 15
display_multi = multi_df.head(15).copy()
display_multi['IS_Sharpe'] = display_multi['IS_Sharpe'].map('{:.3f}'.format)
display_multi['OOS_Sharpe'] = display_multi['OOS_Sharpe'].map('{:.3f}'.format)
display_multi['vs_TEMA_Alone'] = display_multi['vs_TEMA_Alone'].map('{:+.1f}%'.format)
display_multi['Boruta_Score'] = display_multi['Boruta_Score'].map('{:.1f}%'.format)
display_multi['Total_Return'] = display_multi['Total_Return'].map('{:.2%}'.format)
display_multi['Max_DD'] = display_multi['Max_DD'].map('{:.2%}'.format)
display_multi['Win_Rate'] = display_multi['Win_Rate'].map('{:.1%}'.format)

print("\n" + "="*100)
print("📊 TOP 15 MULTI-INDICATOR ENSEMBLES (Ranked by OOS Sharpe)")
print("="*100)
print(display_multi.to_string(index=False))

In [ ]:
# ==================== CELL 10: FINAL RECOMMENDATIONS ====================

print("\n" + "="*100)
print("🏆 FINAL BORUTA FEATURE SELECTION RESULTS")
print("="*100)

# Best Individual Indicator (by Boruta)
best_individual = individual_df.iloc[0]
print(f"\n📍 BEST INDIVIDUAL INDICATOR (by Boruta Score):")
print(f"   Indicator: {best_individual['Indicator']}")
print(f"   IS Sharpe: {best_individual['IS_Sharpe']:.3f}")
print(f"   OOS Sharpe: {best_individual['OOS_Sharpe']:.3f}")
print(f"   Boruta Score: {best_individual['Boruta_Score']:.1f}% {best_individual['Boruta_Confirmed']}")

# Best TEMA + 1 Ensemble
if len(ensemble_df) > 0:
    best_duo = ensemble_df.iloc[0]
    print(f"\n📍 BEST TEMA + 1 ENSEMBLE (by OOS Sharpe):")
    print(f"   Ensemble: {best_duo['Ensemble']}")
    print(f"   IS Sharpe: {best_duo['IS_Sharpe']:.3f}")
    print(f"   OOS Sharpe: {best_duo['OOS_Sharpe']:.3f}")
    print(f"   vs TEMA Alone: {best_duo['vs_TEMA_Alone']:+.1f}%")
    print(f"   Boruta Score: {best_duo['Boruta_Score']:.1f}% {best_duo['Boruta_Confirmed']}")

# Best Multi-Indicator Ensemble
if len(multi_df) > 0:
    best_multi = multi_df.iloc[0]
    print(f"\n📍 BEST MULTI-INDICATOR ENSEMBLE (by OOS Sharpe):")
    print(f"   Ensemble: {best_multi['Ensemble']}")
    print(f"   # Indicators: {best_multi['Num_Indicators']}")
    print(f"   IS Sharpe: {best_multi['IS_Sharpe']:.3f}")
    print(f"   OOS Sharpe: {best_multi['OOS_Sharpe']:.3f}")
    print(f"   vs TEMA Alone: {best_multi['vs_TEMA_Alone']:+.1f}%")
    print(f"   Boruta Score: {best_multi['Boruta_Score']:.1f}% {best_multi['Boruta_Confirmed']}")

# TEMA Baseline for comparison
print(f"\n📍 BASELINE COMPARISON - TEMA ALONE:")
print(f"   IS Sharpe: {tema_is_sharpe:.3f}")
print(f"   OOS Sharpe: {tema_oos_sharpe:.3f}")
print(f"   Boruta Score: {tema_boruta:.1f}%")

print("\n" + "="*100)
print("💡 RECOMMENDATIONS")
print("="*100)

# Identify confirmed ensembles that improve on TEMA
confirmed_improvements = ensemble_df[(ensemble_df['Boruta_Confirmed'] == '✅ YES') & (ensemble_df['vs_TEMA_Alone'] > 0)]

if len(confirmed_improvements) > 0:
    print(f"\n✅ {len(confirmed_improvements)} ensemble(s) IMPROVE on TEMA AND are Boruta-confirmed:")
    for _, row in confirmed_improvements.iterrows():
        print(f"   • {row['Ensemble']}: OOS Sharpe {row['OOS_Sharpe']:.3f} ({row['vs_TEMA_Alone']:+.1f}%)")
else:
    print("\n⚠️ No ensembles both improve OOS Sharpe AND pass Boruta validation.")
    print("   Consider:")
    print("   1. Using TEMA alone if it's Boruta-confirmed")
    print("   2. Testing different indicator parameters")
    print("   3. Using AND logic instead of OR logic")

print("\n" + "="*100)

In [ ]:
# ==================== CELL 11: VISUALIZATION ====================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Individual Indicator Boruta Scores
ax1 = axes[0, 0]
colors = ['green' if x >= 95 else 'orange' if x >= 80 else 'red' for x in individual_df['Boruta_Score']]
bars1 = ax1.barh(individual_df['Indicator'], individual_df['Boruta_Score'], color=colors, edgecolor='black')
ax1.axvline(95, color='green', linestyle='--', linewidth=2, label='95% Threshold')
ax1.axvline(80, color='orange', linestyle='--', linewidth=1.5, label='80% Threshold')
ax1.set_xlabel('Boruta Score (%)', fontweight='bold')
ax1.set_title('Individual Indicator Boruta Scores', fontweight='bold', fontsize=12)
ax1.legend()
ax1.set_xlim(0, 105)

# 2. IS vs OOS Sharpe Comparison
ax2 = axes[0, 1]
x = np.arange(len(individual_df))
width = 0.35
bars2a = ax2.bar(x - width/2, individual_df['IS_Sharpe'], width, label='IS Sharpe', color='blue', alpha=0.7)
bars2b = ax2.bar(x + width/2, individual_df['OOS_Sharpe'], width, label='OOS Sharpe', color='green', alpha=0.7)
ax2.set_xlabel('Indicator', fontweight='bold')
ax2.set_ylabel('Sharpe Ratio', fontweight='bold')
ax2.set_title('In-Sample vs Out-of-Sample Sharpe', fontweight='bold', fontsize=12)
ax2.set_xticks(x)
ax2.set_xticklabels(individual_df['Indicator'], rotation=45, ha='right')
ax2.legend()
ax2.axhline(0, color='black', linewidth=0.5)

# 3. TEMA Ensemble Comparison
ax3 = axes[1, 0]
if len(ensemble_df) > 0:
    ens_names = [e.replace('TEMA OR ', '') for e in ensemble_df['Ensemble']]
    colors3 = ['green' if x >= 0 else 'red' for x in ensemble_df['vs_TEMA_Alone']]
    bars3 = ax3.barh(ens_names, ensemble_df['vs_TEMA_Alone'], color=colors3, edgecolor='black')
    ax3.axvline(0, color='black', linewidth=1.5)
    ax3.set_xlabel('OOS Sharpe Change vs TEMA Alone (%)', fontweight='bold')
    ax3.set_title('Ensemble Improvement over TEMA', fontweight='bold', fontsize=12)

# 4. Risk-Return Scatter
ax4 = axes[1, 1]
scatter = ax4.scatter(individual_df['Max_DD'] * 100, individual_df['OOS_Sharpe'], 
                      c=individual_df['Boruta_Score'], cmap='RdYlGn', 
                      s=150, edgecolors='black', linewidth=1)
for i, txt in enumerate(individual_df['Indicator']):
    ax4.annotate(txt, (individual_df['Max_DD'].iloc[i] * 100, individual_df['OOS_Sharpe'].iloc[i]),
                 fontsize=8, ha='center', va='bottom')
ax4.set_xlabel('Max Drawdown (%)', fontweight='bold')
ax4.set_ylabel('OOS Sharpe Ratio', fontweight='bold')
ax4.set_title('Risk-Return Profile (Color = Boruta Score)', fontweight='bold', fontsize=12)
plt.colorbar(scatter, ax=ax4, label='Boruta Score %')

plt.tight_layout()
plt.savefig('boruta_feature_selection_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to 'boruta_feature_selection_results.png'")

In [ ]:
# ==================== CELL 12: EXPORT RESULTS ====================

# Save all results to CSV for further analysis
individual_df.to_csv('boruta_individual_indicators.csv', index=False)
ensemble_df.to_csv('boruta_tema_ensembles.csv', index=False)
multi_df.to_csv('boruta_multi_ensembles.csv', index=False)

print("="*70)
print("📁 RESULTS EXPORTED")
print("="*70)
print("   • boruta_individual_indicators.csv")
print("   • boruta_tema_ensembles.csv")
print("   • boruta_multi_ensembles.csv")
print("   • boruta_feature_selection_results.png")
print("\n✅ Boruta Feature Selection Complete!")

---
## 📝 How to Interpret Results

### Boruta Score
- **95-100%**: ✅ Strong statistical significance - indicator has real predictive power
- **80-94%**: ⚠️ Moderate significance - may be useful but less reliable
- **<80%**: ❌ Weak/no significance - likely noise or overfitting

### Sharpe Decay
- **Positive decay**: OOS performs BETTER than IS (rare but good)
- **Slight negative (-20% to 0)**: Normal and acceptable
- **Large negative (<-50%)**: Likely overfitting - be cautious

### Ensemble Selection
1. Prefer ensembles where BOTH indicators are Boruta-confirmed
2. Look for positive improvement vs TEMA alone
3. Consider complexity - simpler (fewer indicators) is often better
4. Check that OOS Sharpe > 0.5 for live trading consideration

### Next Steps
1. Run sensitivity analysis on your chosen ensemble (as Rick mentioned)
2. Test on additional OOS periods (walk-forward)
3. Paper trade before going live